# Example: Building a Maximum-Utility Portfolio Allocator
In this example, we translate asset-level preference scores into budget-feasible Cobb–Douglas and CES allocations and examine how the elasticity parameter changes concentration.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct adaptive preference weights:__ Combine simplified SIM reward, market exposure, idiosyncratic risk, and a market-state signal.
> * __Allocate a fixed budget:__ Compute closed-form Cobb–Douglas and CES demands.
> * __Interpret elasticity:__ Explain how the CES elasticity parameter changes substitution and portfolio concentration.

Let's turn a vector of preferences into an auditable allocation.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Construct Preference Weights
We load a frozen 2014–2024 SIM calibration and its paired price snapshot through the course package, select five representative assets, and define a transparent score from the SIM parameters and current market-state signal $\lambda$:
$$
s_i=a\alpha_i-b\lvert\beta_i-1\rvert-c\sigma_{\varepsilon,i}+d\lambda\beta_i.
$$
We convert scores to positive, normalized preference weights with a softmax transformation.


In [ ]:
calibration_snapshot = MySIMCalibration();
price_snapshot = MyCurrentPrices();

tickers = ["AAPL", "AMZN", "JPM", "NVDA", "XOM"];
calibration_index = [findfirst(==(ticker), calibration_snapshot["tickers"]) for ticker in tickers];
price_index = [findfirst(==(ticker), price_snapshot["tickers"]) for ticker in tickers];

prices = price_snapshot["prices"][price_index];
α = calibration_snapshot["alpha"][calibration_index];
β = calibration_snapshot["beta"][calibration_index];
σε = calibration_snapshot["sigma_eps"][calibration_index];
λ = -0.35; # negative denotes a defensive market-state signal

# The frozen calibration uses annualized CCGR units, so its residual-risk scale
# is larger than the small synthetic values used in introductory exercises.
scores = 5 .* α .- 0.8 .* abs.(β .- 1) .- 0.15 .* σε .+ 0.4 .* λ .* β;
γ = adaptive_preference_weights(α, β, σε, λ; alpha_gain=5.0, risk_penalty=0.15);

preference_df = DataFrame(ticker=tickers, price=prices, alpha=α,
    beta=β, residual_risk=σε, score=scores, preference=γ);
pretty_table(preference_df; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 2: Compute the Cobb–Douglas Allocation
For normalized exponents $\sum_i\gamma_i=1$, maximizing
$$
U_{CD}(\mathbf x)=\prod_i x_i^{\gamma_i}
\qquad\text{subject to}\qquad
\sum_i p_i x_i\leq B
$$
gives the closed-form demand $x_i^{\star}=\gamma_iB/p_i$. Consequently, the portfolio budget weights equal the preference weights.


In [ ]:
budget = 100_000.0;
cd = allocate_cobb_douglas(prices, γ, budget);
cd_df = DataFrame(ticker=tickers, shares=cd.shares,
    dollars=cd.dollars, portfolio_weight=cd.weights);
pretty_table(cd_df; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 3: Compare CES Allocations
For elasticity $\eta>0$, the CES demand used here is
$$
x_i^{\star}=
\frac{B(\gamma_i/p_i)^{\eta}}
{\sum_j p_j(\gamma_j/p_j)^{\eta}}.
$$
The case $\eta=1$ recovers the Cobb–Douglas allocation. Larger values concentrate expenditure on assets with more favorable preference-to-price ratios, whereas smaller positive values spread demand more evenly.


In [ ]:
elasticities = [0.5, 1.0, 2.0, 5.0];
comparison = DataFrame(ticker=tickers);
for η in elasticities
    comparison[!, Symbol("eta_$(η)")] = allocate_ces(prices, γ, budget, η).weights;
end

pretty_table(comparison; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
concentration = [sum(allocate_ces(prices, γ, budget, η).weights.^2)
    for η in elasticities];
plot(elasticities, concentration, marker=:circle, lw=2, c=:navy,
    xlabel="CES elasticity η", ylabel="Herfindahl concentration",
    label="Portfolio concentration")


## Summary
This example separated three modeling choices: how information becomes a preference score, how preferences become a feasible allocation, and how elasticity controls substitution.

> __Key Takeaways:__
>
> * __Preference construction is part of the model:__ The allocator cannot repair an opaque or poorly calibrated score.
> * __Cobb–Douglas is transparent:__ Normalized preferences map directly to budget shares.
> * __CES adds a concentration control:__ The elasticity parameter changes how strongly the allocator favors high preference-to-price assets.

The next example places this allocator inside a guarded rebalancing loop.
___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Utility functions and preference scores are simplified representations of investor objectives and constraints.
